In [0]:
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql import Window
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [0]:
df = spark.table("workspace.default.bank_churners")
df.count()

In [0]:
df = df.drop('Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2')

In [0]:
df.printSchema()

In [0]:
# Analisando conteúdo completo
for coluna in df.columns:
  df.groupBy(coluna).count().show(truncate=False)

In [0]:
def completudeVar(df, df_nome="DataFrame"):
    """
    Gera um resumo por coluna mostrando:
    - Qtd de duplicados
    - Qtd de nulos
    - Qtd de valores únicos
    """
    colunas = df.columns
    resumo = []

    for col in colunas:
        total = df.count()
        nulos = df.filter(F.col(col).isNull()).count()
        duplicados = total - df.select(col).distinct().count()
        unicos = df.select(col).distinct().count()

        resumo.append({
            "coluna": col,
            "total": total,
            "duplicados": duplicados,
            "nulos": nulos,
            "valores_unicos": unicos
        })

    resumo_pd = pd.DataFrame(resumo)
    resumo_pd = resumo_pd.sort_values("duplicados", ascending=False).reset_index(drop=True)
    
    print(f"Resumo do {df_nome}:")
    display(resumo_pd)
    return resumo_pd

In [0]:
# Analisando qualidade do dado
completudeVar(df)

### Padronização dos dados para treinamento do modelo
- Transformar todas em minúsculas
- Transforma-las em categoricas

In [0]:
df_padronizado = df.withColumn('attrition_flag', F.when(F.col('attrition_flag') == 'Existing Customer', 0).otherwise(1))
df_padronizado = df_padronizado.withColumn('Gender', F.when(F.col('Gender') == 'Female', 1).otherwise(2))


In [0]:
change_cols = ['Education_Level', 'Income_Category', 'Card_Category', 'Marital_Status']
one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', one_hot_encoder, change_cols)
    ],
    remainder='passthrough' # Mantém as outras colunas (numéricas) intactas
)

df_encoded_array = preprocessor.fit_transform(df_padronizado)

nomes_novas_colunas = preprocessor.get_feature_names_out()
df_encoded_ct = pd.DataFrame(df_encoded_array, columns=nomes_novas_colunas)

print(df_encoded_ct)


In [0]:
### Validação das mudanças
df.groupBy('af_binary', "Attrition_Flag").count().show()

df.groupBy('af_binary', "Attrition_Flag").count().show()


Logistic Regression